# Transfer MR Exams & ECC Data: Bastion → Fourier

Staged notebook to:
1. Parse the Excel manifest
2. Resolve source paths on bastion (handling `-N` suffixes, multi-PID patients)
3. Build and validate a transfer manifest
4. Generate an rsync shell script for the actual transfer

**Sources:**
- MR exams: `/media/evmasuta/hdd/cardiac-ucsd-mr-exams/`
- ECC velocities: `/media/evmasuta/hdd/cardiac-ucsd-ecc-npy/`

**Destinations:**
- MR exams: `/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/`
- ECC velocities: `/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/corrected_velocities/`

In [1]:
import os
import re
import json
from pathlib import Path
from dataclasses import dataclass, field, asdict
from typing import List, Dict, Optional, Tuple

import pandas as pd

# === Paths ===
EXCEL_PATH = Path("/mnt/yeluru/vascular-superenhancement-4d-flow/splits/ecc_data_4dflow_3dcine_pids (2).xlsx")
MR_SOURCE = Path("/media/evmasuta/hdd/cardiac-ucsd-mr-exams")
ECC_SOURCE = Path("/media/evmasuta/hdd/cardiac-ucsd-ecc-npy")
MR_DEST = Path("/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files")
ECC_DEST = Path("/home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/corrected_velocities")

assert EXCEL_PATH.exists(), f"Excel not found: {EXCEL_PATH}"
assert MR_SOURCE.exists(), f"MR source not found: {MR_SOURCE}"
assert ECC_SOURCE.exists(), f"ECC source not found: {ECC_SOURCE}"
assert MR_DEST.exists(), f"MR dest not found: {MR_DEST}"
assert ECC_DEST.exists(), f"ECC dest not found: {ECC_DEST}"

print("All paths verified.")

All paths verified.


## Stage 1: Parse Excel

In [2]:
def parse_series_column(text: str) -> Dict[str, List[str]]:
    """
    Parse a series column like 'Dehujar: 4090, 4091; Fuquedoy: 4090'
    into {'Dehujar': ['4090','4091'], 'Fuquedoy': ['4090']}.
    A trailing colon with nothing after means empty series list.
    """
    result = {}
    if not text or str(text).strip() == "" or text != text:  # last check catches NaN
        return result
    parts = str(text).split(";")
    for part in parts:
        part = part.strip()
        if ":" not in part:
            continue
        name, series_str = part.split(":", 1)
        name = name.strip()
        series_str = series_str.strip()
        series = [s.strip() for s in series_str.split(",") if s.strip()] if series_str else []
        result[name] = series
    return result


@dataclass
class PatientRecord:
    accession: int
    pids: List[str]
    study_keys: List[str]
    cine3d_series: Dict[str, List[str]]
    flow4d_series: Dict[str, List[str]]
    flow4d_pid: Optional[str] = None


def parse_excel(path: Path) -> List[PatientRecord]:
    df = pd.read_excel(path, sheet_name="Accession_Summary")
    records = []
    for _, row in df.iterrows():
        acc = row["Accession"]
        pids_str = row["Affiliated PIDs"]
        keys_str = row["Affiliated Study Keys"]
        cine3d_str = row.get("3dcine", "")
        flow4d_str = row.get("4dflow", "")

        if pd.isna(acc) or pd.isna(pids_str):
            continue

        pids = [p.strip() for p in str(pids_str).split(";")]
        keys = [k.strip() for k in str(keys_str).split(";")]
        cine3d = parse_series_column(cine3d_str)
        flow4d = parse_series_column(flow4d_str)

        # 4dflow PID must have >= 4 series (mag + 3 velocity components; 5 if scout included)
        # A PID with only 1 series is likely just a scout sequence
        best_pid = None
        best_count = 0
        for pid in pids:
            count = len(flow4d.get(pid, []))
            if count >= 4 and count > best_count:
                best_count = count
                best_pid = pid

        records.append(PatientRecord(
            accession=int(acc),
            pids=pids,
            study_keys=keys,
            cine3d_series=cine3d,
            flow4d_series=flow4d,
            flow4d_pid=best_pid,
        ))
    return records


records = parse_excel(EXCEL_PATH)
print(f"Parsed {len(records)} patient records from Excel.")
print(f"  Multi-PID: {sum(1 for r in records if len(r.pids) > 1)}")
print(f"  Single-PID: {sum(1 for r in records if len(r.pids) == 1)}")
print(f"  Has 4dflow PID (>=4 series): {sum(1 for r in records if r.flow4d_pid)}")
print(f"  No qualifying 4dflow PID: {sum(1 for r in records if not r.flow4d_pid)}")

# Show distribution of max 4dflow series count per patient
from collections import Counter
max_flow_counts = Counter()
for r in records:
    max_count = max((len(s) for s in r.flow4d_series.values()), default=0)
    max_flow_counts[max_count] += 1
print(f"\n  4dflow series count distribution (max across PIDs per patient):")
for count in sorted(max_flow_counts.keys()):
    print(f"    {count} series: {max_flow_counts[count]} patients")

# Collect all series numbers per patient across modalities
has_3dcine_strict = 0
no_3dcine = 0
has_both = 0
for r in records:
    all_flow_series = set()
    for s_list in r.flow4d_series.values():
        all_flow_series.update(s_list)
    all_cine_series = set()
    for s_list in r.cine3d_series.values():
        all_cine_series.update(s_list)

    # 3dcine series that are NOT also listed as 4dflow series
    unique_cine = all_cine_series - all_flow_series
    if unique_cine:
        has_3dcine_strict += 1
        if r.flow4d_pid:
            has_both += 1
    else:
        no_3dcine += 1

print(f"  Has 3dcine (series distinct from 4dflow): {has_3dcine_strict}")
print(f"  No distinct 3dcine series: {no_3dcine}")
print(f"  Has BOTH 4dflow + distinct 3dcine: {has_both}")

# Show a few examples to sanity-check
print("\n=== Sample patients WITH distinct 3dcine ===")
count = 0
for r in records:
    all_flow = set()
    for s in r.flow4d_series.values(): all_flow.update(s)
    all_cine = set()
    for s in r.cine3d_series.values(): all_cine.update(s)
    unique_cine = all_cine - all_flow
    if unique_cine and count < 5:
        print(f"  Acc={r.accession}: cine={r.cine3d_series}, flow={r.flow4d_series}, unique_cine={unique_cine}")
        count += 1

print("\n=== Sample patients where cine overlaps with 4dflow (not counted) ===")
count = 0
for r in records:
    all_flow = set()
    for s in r.flow4d_series.values(): all_flow.update(s)
    all_cine = set()
    for s in r.cine3d_series.values(): all_cine.update(s)
    if all_cine and not (all_cine - all_flow) and count < 5:
        print(f"  Acc={r.accession}: cine={r.cine3d_series}, flow={r.flow4d_series}")
        count += 1

Parsed 547 patient records from Excel.
  Multi-PID: 222
  Single-PID: 325
  Has 4dflow PID (>=4 series): 511
  No qualifying 4dflow PID: 36

  4dflow series count distribution (max across PIDs per patient):
    0 series: 6 patients
    1 series: 30 patients
    4 series: 6 patients
    5 series: 482 patients
    6 series: 2 patients
    7 series: 18 patients
    8 series: 2 patients
    10 series: 1 patients
  Has 3dcine (series distinct from 4dflow): 464
  No distinct 3dcine series: 83
  Has BOTH 4dflow + distinct 3dcine: 436

=== Sample patients WITH distinct 3dcine ===
  Acc=43671239: cine={'Upoktib': ['1400']}, flow={'Upoktib': ['17', '1700', '1701', '1702', '1703']}, unique_cine={'1400'}
  Acc=43759484: cine={'Kecuesor': ['10', '1000'], 'Kidimmem': ['10', '1000']}, flow={'Kecuesor': [], 'Kidimmem': []}, unique_cine={'10', '1000'}
  Acc=43803013: cine={'Dehujar': [], 'Fuquedoy': ['7', '700']}, flow={'Dehujar': ['4090', '4091', '4092', '4093', '4094'], 'Fuquedoy': ['4090']}, unique_

In [3]:
# Show representative examples from each category

# Standard case: single PID, 5 4dflow series, has 3dcine
print("=== Standard: single PID, 5 flow series + 3dcine ===")
for r in records:
    if len(r.pids) == 1 and r.flow4d_pid and len(r.flow4d_series.get(r.flow4d_pid, [])) == 5:
        all_cine = set()
        for s in r.cine3d_series.values(): all_cine.update(s)
        all_flow = set()
        for s in r.flow4d_series.values(): all_flow.update(s)
        if all_cine - all_flow:
            print(f"  Acc={r.accession}, PID={r.pids[0]}, flow={r.flow4d_series}, cine={r.cine3d_series}")
            break

# Multi-PID: one has 4dflow, other has 3dcine
print("\n=== Multi-PID: one PID has flow, other has cine ===")
for r in records:
    if len(r.pids) == 2 and r.flow4d_pid:
        other = [p for p in r.pids if p != r.flow4d_pid][0]
        if r.cine3d_series.get(other, []) and not r.flow4d_series.get(other, []):
            print(f"  Acc={r.accession}, PIDs={r.pids}, 4dflow_pid={r.flow4d_pid}")
            print(f"    flow={r.flow4d_series}, cine={r.cine3d_series}")
            break

# Scout only: 1 flow series, no qualifying 4dflow PID
print("\n=== Scout only: 1 flow series, no qualifying 4dflow PID ===")
for r in records[:]:
    if not r.flow4d_pid:
        max_count = max((len(s) for s in r.flow4d_series.values()), default=0)
        if max_count == 1:
            print(f"  Acc={r.accession}, PIDs={r.pids}, flow={r.flow4d_series}")
            break

# 4 flow series (no scout)
print("\n=== 4 flow series (mag + 3 velocity, no scout) ===")
for r in records:
    if r.flow4d_pid and len(r.flow4d_series.get(r.flow4d_pid, [])) == 4:
        print(f"  Acc={r.accession}, PID={r.flow4d_pid}, flow={r.flow4d_series}, cine={r.cine3d_series}")
        break

# 6+ flow series (repeated acquisition)
print("\n=== 6+ flow series (repeated/extra acquisitions) ===")
for r in records:
    if r.flow4d_pid and len(r.flow4d_series.get(r.flow4d_pid, [])) >= 6:
        print(f"  Acc={r.accession}, PID={r.flow4d_pid}, flow={r.flow4d_series}, cine={r.cine3d_series}")
        break

# No 4dflow at all (0 series)
print("\n=== No 4dflow data at all (0 series across all PIDs) ===")
for r in records:
    if all(len(s) == 0 for s in r.flow4d_series.values()):
        print(f"  Acc={r.accession}, PIDs={r.pids}, flow={r.flow4d_series}, cine={r.cine3d_series}")
        break

=== Standard: single PID, 5 flow series + 3dcine ===
  Acc=43671239, PID=Upoktib, flow={'Upoktib': ['17', '1700', '1701', '1702', '1703']}, cine={'Upoktib': ['1400']}

=== Multi-PID: one PID has flow, other has cine ===
  Acc=43856544, PIDs=['Quigotab', 'Uragog'], 4dflow_pid=Quigotab
    flow={'Quigotab': ['4090', '4091', '4092', '4093', '4094'], 'Uragog': []}, cine={'Quigotab': [], 'Uragog': ['700']}

=== Scout only: 1 flow series, no qualifying 4dflow PID ===
  Acc=43194035, PIDs=['Ifiedus'], flow={'Ifiedus': ['4230']}

=== 4 flow series (mag + 3 velocity, no scout) ===
  Acc=51367711, PID=Fajupo, flow={'Fajupo': ['4130', '4131', '4133', '4134']}, cine={'Fajupo': ['12', '1200']}

=== 6+ flow series (repeated/extra acquisitions) ===
  Acc=50599348, PID=Strejekast, flow={'Kupehu': ['4130'], 'Pecosoy': [], 'Strejekast': ['2400', '2401', '2402', '2403', '2404', '2405', '2406'], 'Yihero': ['4130', '4131', '4132', '4133', '4134']}, cine={'Kupehu': ['11', '1100'], 'Pecosoy': ['11', '1100'],

## Stage 1.5: Scan DICOM headers to build StudyInstanceUID map

Scans **all** folders in the MR source directory (not just PID-matching ones) to read one DICOM
file from each study subfolder and extract the actual `StudyInstanceUID`. This ensures we find
studies even if they're stored under a folder name that doesn't match the expected PID.
Results are cached to `working_dir/dicom_study_map.csv`.

In [4]:
import pydicom
from tqdm.notebook import tqdm

CACHE_PATH = Path("/mnt/yeluru/vascular-superenhancement-4d-flow/working_dir/dicom_study_map.csv")


def find_first_dicom(folder: Path, max_depth: int = 3) -> Optional[Path]:
    """Walk into a folder up to max_depth levels to find a regular file (DICOM)."""
    for f in folder.rglob("*"):
        if f.is_file():
            return f
    return None


def read_study_uid(dcm_file: Path) -> str:
    try:
        ds = pydicom.dcmread(str(dcm_file), stop_before_pixels=True)
        return str(ds.StudyInstanceUID) if hasattr(ds, "StudyInstanceUID") else "__MISSING_TAG__"
    except Exception as ex:
        return f"__ERROR__:{ex}"


def scan_pid_folder(pid_folder: Path) -> List[Dict[str, str]]:
    """
    For a PID folder, iterate its study subfolders, read one DICOM from each,
    and return a list of {folder_name, subfolder_name, study_instance_uid}.
    Also handles flat folders where DICOMs sit directly in the PID folder.
    """
    results = []
    if not pid_folder.is_dir():
        return results

    has_subdirs = False
    has_files = False
    for entry in sorted(pid_folder.iterdir()):
        if entry.is_dir():
            has_subdirs = True
            dcm_file = find_first_dicom(entry)
            if dcm_file is None:
                uid = "__NO_DICOM_FOUND__"
            else:
                uid = read_study_uid(dcm_file)
            results.append({
                "folder_name": pid_folder.name,
                "subfolder_name": entry.name,
                "study_instance_uid": uid,
            })
        elif entry.is_file():
            has_files = True

    # Flat folder: DICOMs directly in the PID folder, no subdirectories.
    # Sample files to find all unique study UIDs (could be mixed studies).
    if not has_subdirs and has_files:
        seen_uids = set()
        files = [f for f in sorted(pid_folder.iterdir()) if f.is_file()]
        # Sample evenly: first, last, and evenly spaced in between
        sample_indices = set([0, len(files) - 1])
        step = max(1, len(files) // 20)
        sample_indices.update(range(0, len(files), step))
        for idx in sorted(sample_indices):
            uid = read_study_uid(files[idx])
            if uid.startswith("__"):
                continue
            if uid not in seen_uids:
                seen_uids.add(uid)
                results.append({
                    "folder_name": pid_folder.name,
                    "subfolder_name": "__FLAT__",
                    "study_instance_uid": uid,
                })

    return results


def build_dicom_map(mr_source: Path, cache_path: Path) -> pd.DataFrame:
    if cache_path.exists():
        print(f"Loading cached DICOM map from {cache_path}")
        return pd.read_csv(cache_path)

    all_dirs = sorted(d for d in mr_source.iterdir() if d.is_dir())

    rows = []
    for folder in tqdm(all_dirs, desc="Scanning DICOM folders"):
        scan_results = scan_pid_folder(folder)
        for sr in scan_results:
            rows.append(sr)

    df = pd.DataFrame(rows)
    cache_path.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(cache_path, index=False)
    print(f"Saved DICOM map to {cache_path} ({len(df)} rows)")
    return df


dicom_map = build_dicom_map(MR_SOURCE, CACHE_PATH)
print(f"\nDICOM map: {len(dicom_map)} entries across {dicom_map['folder_name'].nunique()} folders")
print(f"\nSample entries:")
print(dicom_map.head(10).to_string(index=False))

Scanning DICOM folders:   0%|          | 0/1590 [00:00<?, ?it/s]

Saved DICOM map to /mnt/yeluru/vascular-superenhancement-4d-flow/working_dir/dicom_study_map.csv (3955 rows)

DICOM map: 3955 entries across 1588 folders

Sample entries:
folder_name                              subfolder_name                                               study_instance_uid
     Abamab                                    __FLAT__ 1.2.826.0.1.3680043.2.1143.8756563745501141171565121332855313610
     Abefel 2.25.78364570408460610489179191601659995621                      2.25.78364570408460610489179191601659995621
 Abscipnuck                 up-5f881132b711ed08287b281b                      2.25.71556380484251089036970727233411611133
   Achelney               2018May17_Ex2576_4dflow_Ser13 1.2.826.0.1.3680043.2.1143.6197375534621376512722992084143177330
Ackdradum-0                 up-5cf04d70f17519787d061202                     2.25.208094282930002299828629468361737004858
Ackdradum-1                 up-5cf04dfb71f65e658a134a3e                     2.25.2080942829300022998286

## Stage 2: Build one-row-per-patient table

For each patient, resolve:
- **4dflow**: PID, study UID, series numbers, source folder on bastion
- **3dcine**: PID, study UID, series numbers, source folder on bastion (if distinct from 4dflow)
- **ECC**: `.npy` file path on bastion

Source folders are matched via the DICOM `StudyInstanceUID` map built in Stage 1.5.

In [5]:
def find_study_folder(study_key: str, mr_source: Path, dcm_map: pd.DataFrame) -> Tuple[Optional[str], Optional[str]]:
    """
    Search the entire DICOM map for a study UID match.
    Returns (full_path, folder_name) or (None, None).
    """
    matched = dcm_map[dcm_map["study_instance_uid"] == study_key]
    if matched.empty:
        return None, None
    folder_name = matched.iloc[0]["folder_name"]
    return str(mr_source / folder_name), folder_name


def build_patient_table(
    records: List[PatientRecord],
    mr_source: Path,
    ecc_source: Path,
    dcm_map: pd.DataFrame,
) -> pd.DataFrame:
    rows = []
    for rec in records:
        # Identify 3dcine PID: the PID with the most distinct (non-4dflow) cine series
        all_flow_series = set()
        for s in rec.flow4d_series.values():
            all_flow_series.update(s)

        cine3d_pid = None
        cine3d_key = None
        cine3d_series = []
        best_cine_count = 0
        for pid, study_key in zip(rec.pids, rec.study_keys):
            cine_series = rec.cine3d_series.get(pid, [])
            unique_cine = [s for s in cine_series if s not in all_flow_series]
            if len(unique_cine) > best_cine_count:
                best_cine_count = len(unique_cine)
                cine3d_pid = pid
                cine3d_key = study_key
                cine3d_series = unique_cine

        # Get study keys for 4dflow PID
        flow4d_key = None
        flow4d_series = []
        if rec.flow4d_pid:
            idx = rec.pids.index(rec.flow4d_pid) if rec.flow4d_pid in rec.pids else None
            if idx is not None:
                flow4d_key = rec.study_keys[idx]
            flow4d_series = rec.flow4d_series.get(rec.flow4d_pid, [])

        # Resolve source paths
        flow4d_src, flow4d_folder = None, None
        if rec.flow4d_pid and flow4d_key:
            flow4d_src, flow4d_folder = find_study_folder(flow4d_key, mr_source, dcm_map)

        cine3d_src, cine3d_folder = None, None
        if cine3d_pid and cine3d_key:
            cine3d_src, cine3d_folder = find_study_folder(cine3d_key, mr_source, dcm_map)

        ecc_path = None
        if rec.flow4d_pid:
            npy = ecc_source / f"{rec.flow4d_pid}.npy"
            if npy.is_file():
                ecc_path = str(npy)

        rows.append({
            "accession": rec.accession,
            "flow4d_pid": rec.flow4d_pid,
            "flow4d_study_uid": flow4d_key,
            "flow4d_series": flow4d_series,
            "flow4d_src_folder": flow4d_folder,
            "cine3d_pid": cine3d_pid if best_cine_count > 0 else None,
            "cine3d_study_uid": cine3d_key if best_cine_count > 0 else None,
            "cine3d_series": cine3d_series if best_cine_count > 0 else [],
            "cine3d_src_folder": cine3d_folder,
            "ecc_npy_name": f"{rec.flow4d_pid}.npy" if rec.flow4d_pid else None,
            "flow4d_src_path": flow4d_src,
            "cine3d_src_path": cine3d_src,
            "ecc_src_path": ecc_path,
        })

    return pd.DataFrame(rows)


patients = build_patient_table(records, MR_SOURCE, ECC_SOURCE, dicom_map)
print(f"Built patient table: {len(patients)} rows")
print(f"\nSample rows:")
print(patients.head(5).to_string(index=False))

Built patient table: 547 rows

Sample rows:
 accession flow4d_pid                            flow4d_study_uid                flow4d_series flow4d_src_folder cine3d_pid                            cine3d_study_uid cine3d_series cine3d_src_folder ecc_npy_name                                     flow4d_src_path                                     cine3d_src_path                                         ecc_src_path
  43194035       None                                        None                           []              None       None                                        None            []              None         None                                                None                                                None                                                 None
  43255776       None                                        None                           []              None       None                                        None            []              None         None    

In [6]:
# === Patient-level bucket summary ===
p = patients  # shorthand

has_flow4d = p["flow4d_pid"].notna()
has_cine3d = p["cine3d_pid"].notna()
has_flow4d_src = p["flow4d_src_path"].notna()
has_cine3d_src = p["cine3d_src_path"].notna()
has_ecc = p["ecc_src_path"].notna()

print(f"Total patients: {len(p)}")
print(f"\n--- Excel-level info ---")
print(f"  Has qualifying 4dflow PID (>=4 series): {has_flow4d.sum()}")
print(f"  No qualifying 4dflow PID:               {(~has_flow4d).sum()}")
print(f"  Has distinct 3dcine PID:                {has_cine3d.sum()}")
print(f"  No distinct 3dcine:                     {(~has_cine3d).sum()}")

print(f"\n--- Bastion source file availability (of {has_flow4d.sum()} with 4dflow PID) ---")
q = p[has_flow4d]
qf = q["flow4d_src_path"].notna()
qc = q["cine3d_src_path"].notna()
qe = q["ecc_src_path"].notna()
q_has_cine_pid = q["cine3d_pid"].notna()

print(f"  4dflow MR folder found:    {qf.sum()}")
print(f"  4dflow MR folder missing:  {(~qf).sum()}")
print(f"  3dcine MR folder found:    {qc.sum()}  (of {q_has_cine_pid.sum()} with 3dcine in excel)")
print(f"  3dcine MR folder missing:  {(q_has_cine_pid & ~qc).sum()}")
print(f"  ECC .npy found:            {qe.sum()}")
print(f"  ECC .npy missing:          {(~qe).sum()}")

print(f"\n--- Bucket breakdown (of {has_flow4d.sum()} with 4dflow PID) ---")
cat = []
for _, row in q.iterrows():
    parts = []
    if pd.notna(row["flow4d_src_path"]): parts.append("flow_mr")
    if pd.notna(row["cine3d_src_path"]): parts.append("cine_mr")
    if pd.notna(row["ecc_src_path"]):    parts.append("ecc")
    cat.append("+".join(parts) if parts else "nothing")

from collections import Counter
bucket_counts = Counter(cat)
for bucket, cnt in sorted(bucket_counts.items(), key=lambda x: -x[1]):
    print(f"  {bucket:30s} {cnt}")

print(f"\n--- Sample rows ---")
display_cols = ["accession", "flow4d_pid", "cine3d_pid", "flow4d_src_path", "cine3d_src_path", "ecc_src_path"]
print(p[display_cols].head(10).to_string(index=False))

Total patients: 547

--- Excel-level info ---
  Has qualifying 4dflow PID (>=4 series): 511
  No qualifying 4dflow PID:               36
  Has distinct 3dcine PID:                464
  No distinct 3dcine:                     83

--- Bastion source file availability (of 511 with 4dflow PID) ---
  4dflow MR folder found:    416
  4dflow MR folder missing:  95
  3dcine MR folder found:    358  (of 436 with 3dcine in excel)
  3dcine MR folder missing:  78
  ECC .npy found:            506
  ECC .npy missing:          5

--- Bucket breakdown (of 511 with 4dflow PID) ---
  flow_mr+cine_mr+ecc            349
  ecc                            88
  flow_mr+ecc                    64
  cine_mr+ecc                    5
  flow_mr+cine_mr                3
  nothing                        1
  cine_mr                        1

--- Sample rows ---
 accession flow4d_pid cine3d_pid                                     flow4d_src_path                                     cine3d_src_path                       

In [7]:
# === Patients with qualifying 4dflow PID but no MR folder on bastion ===
missing_mr = patients[patients["flow4d_pid"].notna() & patients["flow4d_src_path"].isna()]
print(f"{len(missing_mr)} patients have a qualifying 4dflow PID but no MR folder found on bastion:\n")

w_acc, w_pid, w_ecc = 10, 14, 5
hdr = f"{'Accession':>{w_acc}}  {'4dflow PID':<{w_pid}}  {'ECC':>{w_ecc}}"
print(hdr)
print("-" * len(hdr))
for _, row in missing_mr.iterrows():
    ecc = "yes" if pd.notna(row["ecc_src_path"]) else "---"
    print(f"{row['accession']:>{w_acc}}  {row['flow4d_pid']:<{w_pid}}  {ecc:>{w_ecc}}")

# === Transferable patients: per-patient detail view ===
# Requires BOTH 4dflow MR folder AND ECC .npy on bastion
print(f"\n{'='*60}\n")
transferable = patients[patients["flow4d_src_path"].notna() & patients["ecc_src_path"].notna()]
print(f"Transferable patients: {len(transferable)}\n")

# Column widths
w_acc = 10
w_flow_pid = 14
w_cine_pid = 14
w_flow_mr = 7
w_cine_mr = 7
w_ecc = 5

header = (
    f"{'Accession':>{w_acc}}  "
    f"{'4dflow PID':<{w_flow_pid}}  "
    f"{'3dcine PID':<{w_cine_pid}}  "
    f"{'FlowMR':>{w_flow_mr}}  "
    f"{'CineMR':>{w_cine_mr}}  "
    f"{'ECC':>{w_ecc}}"
)
print(header)
print("-" * len(header))

for _, row in transferable.iterrows():
    flow_mr = "yes" if pd.notna(row["flow4d_src_path"]) else "---"
    cine_mr = "yes" if pd.notna(row["cine3d_src_path"]) else "---"
    ecc     = "yes" if pd.notna(row["ecc_src_path"])     else "---"
    cine_pid = row["cine3d_pid"] if pd.notna(row["cine3d_pid"]) else "---"

    print(
        f"{row['accession']:>{w_acc}}  "
        f"{row['flow4d_pid']:<{w_flow_pid}}  "
        f"{cine_pid:<{w_cine_pid}}  "
        f"{flow_mr:>{w_flow_mr}}  "
        f"{cine_mr:>{w_cine_mr}}  "
        f"{ecc:>{w_ecc}}"
    )

95 patients have a qualifying 4dflow PID but no MR folder found on bastion:

 Accession  4dflow PID        ECC
---------------------------------
  43803013  Dehujar           yes
  43858556  Guefackong        yes
  43872096  Igongep           yes
  43904700  Gifodien          yes
  43908573  Drunsterdoch      yes
  43930853  Lequego           yes
  43934296  Gonufi            yes
  43955256  Cugatu            yes
  43967395  Belaque           yes
  43971544  Quaceja           yes
  43984351  Ifamar            yes
  43985119  Fesemab           yes
  43986020  Kusepee           yes
  43988440  Rosomo            yes
  44005401  Sotomaln          yes
  44014935  Alokin            yes
  44015123  Stipemaf          yes
  44033013  Rujehi            yes
  44048481  Jabepoun          yes
  44053831  Nakodi            yes
  44055587  Kesalo            yes
  44062644  Egiseg            yes
  50659711  Donasnue          ---
  50662383  Bahibo            ---
  50918770  Sapada            yes
  517

In [8]:
# === Patients with no qualifying 4dflow PID ===
no4d = patients[patients["flow4d_pid"].isna()]
print(f"{len(no4d)} patients with no qualifying 4dflow PID (<4 series):\n")
print(no4d[["accession", "cine3d_pid", "cine3d_series"]].to_string(index=False))

36 patients with no qualifying 4dflow PID (<4 series):

 accession cine3d_pid        cine3d_series
  43194035       None                   []
  43255776       None                   []
  43340818       None                   []
  43592613       None                   []
  43695181       None                   []
  43759484   Kecuesor           [10, 1000]
  51982421       None                   []
  51987506       None                   []
  52205960    Woduree           [11, 1100]
  52263691  Jougbaras           [11, 1100]
  52522390   Bonmosin           [18, 1800]
  52589944    Eskinop           [11, 1100]
  52697818   Leyiemuk           [21, 2100]
  52767784    Dequema           [13, 1300]
  52776695     Lifono           [12, 1200]
  52797189     Tutife               [1600]
  52798422    Gonoquo           [15, 1500]
  52832496    Usepjem           [13, 1300]
  52834433     Tajehi           [12, 1200]
  52848803   Boofeque           [19, 1900]
  52849808       None                   [

In [9]:
# === Full patient table export (for inspection / downstream use) ===
export_path = Path("/mnt/yeluru/vascular-superenhancement-4d-flow/splits/ecc_patient_manifest_4dflow_3dcine_bastion.csv")

export_df = patients.copy()
export_df["flow4d_series"] = export_df["flow4d_series"].apply(lambda x: ";".join(x) if isinstance(x, list) else "")
export_df["cine3d_series"] = export_df["cine3d_series"].apply(lambda x: ";".join(x) if isinstance(x, list) else "")
export_df.to_csv(export_path, index=False)
print(f"Exported patient manifest to {export_path} ({len(export_df)} rows)")

Exported patient manifest to /mnt/yeluru/vascular-superenhancement-4d-flow/splits/ecc_patient_manifest_4dflow_3dcine_bastion.csv (547 rows)


## Stage 3: Validate — check for collisions and preview

Before copying, verify:
- No destination folder already exists with the same name
- No duplicate 4dflow PIDs across different accessions

In [10]:
from collections import Counter

# Check for duplicate 4dflow PIDs
pid_counts = patients["flow4d_pid"].dropna().value_counts()
duplicates = pid_counts[pid_counts > 1]
if len(duplicates):
    print(f"WARNING: {len(duplicates)} duplicate 4dflow PIDs across accessions:")
    for pid, cnt in duplicates.items():
        accs = patients.loc[patients["flow4d_pid"] == pid, "accession"].tolist()
        print(f"  {pid}: used in accessions {accs}")
else:
    print("No duplicate 4dflow PIDs — good.")

# Check for collisions with existing destination
existing_mr = set(d.name for d in MR_DEST.iterdir() if d.is_dir())
existing_ecc = set(f.name for f in ECC_DEST.iterdir() if f.is_file())

mr_collisions = patients.loc[patients["flow4d_pid"].isin(existing_mr), "flow4d_pid"].tolist()
ecc_collisions = patients.loc[patients["ecc_npy_name"].isin(existing_ecc), "flow4d_pid"].tolist()

print(f"\nMR destination collisions: {len(mr_collisions)}")
if mr_collisions:
    print(f"  {mr_collisions[:10]}...")
print(f"ECC destination collisions: {len(ecc_collisions)}")
if ecc_collisions:
    print(f"  {ecc_collisions[:10]}...")

No duplicate 4dflow PIDs — good.

MR destination collisions: 333
  ['Upoktib', 'Nokoupa', 'Ipunkan', 'Mabaydem', 'Nadistoub', 'Begaca', 'Voufafu', 'Josipop', 'Negejik', 'Kihoukup']...
ECC destination collisions: 0


In [11]:
# === Build transfer operation lists from the patient table ===
# Requires BOTH 4dflow MR folder AND ECC .npy
actionable = patients[patients["flow4d_src_path"].notna() & patients["ecc_src_path"].notna()]
print(f"{len(actionable)} patients with transferable data\n")

flow_mr_ops = []  # (src_folder, dest_folder) — 4dflow MR
cine_mr_ops = []  # (src_folder, dest_folder) — 3dcine MR (different PID folder)
ecc_ops = []      # (src_file, dest_file)

for _, row in actionable.iterrows():
    dest_name = row["flow4d_pid"]
    dest_mr = str(MR_DEST / dest_name)

    flow_mr_ops.append((row["flow4d_src_path"], dest_mr))

    if pd.notna(row["cine3d_src_path"]) and row["cine3d_src_path"] != row["flow4d_src_path"]:
        cine_mr_ops.append((row["cine3d_src_path"], dest_mr))

    ecc_ops.append((row["ecc_src_path"], str(ECC_DEST / f"{dest_name}.npy")))

mr_ops = flow_mr_ops + cine_mr_ops

print(f"4dflow MR folder ops: {len(flow_mr_ops)}")
print(f"3dcine MR folder ops: {len(cine_mr_ops)}  (different PID folder from 4dflow)")
print(f"ECC .npy ops:         {len(ecc_ops)}")
print(f"Total MR ops:         {len(mr_ops)}")

print("\n=== First 10 4dflow MR ops ===")
for src, dst in flow_mr_ops[:10]:
    print(f"  {src}  -->  {dst}")

print("\n=== All 3dcine MR ops (different PID folder) ===")
for src, dst in cine_mr_ops:
    print(f"  {src}  -->  {dst}")

print("\n=== First 10 ECC ops ===")
for src, dst in ecc_ops[:10]:
    print(f"  {src}  -->  {dst}")

413 patients with transferable data

4dflow MR folder ops: 413
3dcine MR folder ops: 172  (different PID folder from 4dflow)
ECC .npy ops:         413
Total MR ops:         585

=== First 10 4dflow MR ops ===
  /media/evmasuta/hdd/cardiac-ucsd-mr-exams/Upoktib-0  -->  /home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/Upoktib
  /media/evmasuta/hdd/cardiac-ucsd-mr-exams/Nokoupa-0  -->  /home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/Nokoupa
  /media/evmasuta/hdd/cardiac-ucsd-mr-exams/Ipunkan-0  -->  /home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/Ipunkan
  /media/evmasuta/hdd/cardiac-ucsd-mr-exams/Quigotab  -->  /home/ayeluru/mnt/fourier/repository/vascular-superenhancement-4d-flow/all_patients/unzipped_files/Quigotab
  /media/evmasuta/hdd/cardiac-ucsd-mr-exams/Mabaydem-0  -->  /home/ayeluru/mnt/fourier/repository/vascular-superenhancement

## Stage 4: Generate rsync script

**Only run the cells below once you're satisfied with the manifest above.**

Generates a shell script with rsync commands that you can run from the terminal.
For multi-PID patients where multiple source folders map to one destination,
separate rsync lines are emitted so their study-key subfolders merge into one folder.

In [12]:
# === Generate rsync script ===
from collections import defaultdict

SCRIPT_DIR = Path("/mnt/yeluru/vascular-superenhancement-4d-flow/working_dir")
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
rsync_script = SCRIPT_DIR / "rsync_transfer.sh"

dest_to_sources = defaultdict(list)
for src, dst in mr_ops:
    dest_to_sources[dst].append(src)

total_ops = len(mr_ops) + len(ecc_ops)

lines = [
    "#!/bin/bash",
    "",
    "FAIL=0",
    "OK=0",
    f"TOTAL={total_ops}",
    "",
    "# === MR exam folder transfers ===",
    f"# {len(dest_to_sources)} destination folders from {len(mr_ops)} source folders",
    "",
]

op_idx = 0
for dst, srcs in dest_to_sources.items():
    lines.append(f"mkdir -p '{dst}'")
    for src in srcs:
        op_idx += 1
        src_name = Path(src).name
        dst_name = Path(dst).name
        lines.append(f"echo '[{op_idx}/{total_ops}] MR: {src_name} -> {dst_name}'")
        lines.append(f"if rsync -aq '{src}/' '{dst}/'; then")
        lines.append(f"  ((OK++))")
        lines.append(f"else")
        lines.append(f"  echo 'FAILED: {src} -> {dst}' >&2")
        lines.append(f"  ((FAIL++))")
        lines.append(f"fi")

lines.append("")
lines.append("# === ECC .npy file transfers ===")
lines.append(f"# {len(ecc_ops)} files")
lines.append("")

for src, dst in ecc_ops:
    op_idx += 1
    src_name = Path(src).name
    lines.append(f"echo '[{op_idx}/{total_ops}] ECC: {src_name}'")
    lines.append(f"if rsync -aq '{src}' '{dst}'; then")
    lines.append(f"  ((OK++))")
    lines.append(f"else")
    lines.append(f"  echo 'FAILED: {src} -> {dst}' >&2")
    lines.append(f"  ((FAIL++))")
    lines.append(f"fi")

lines.append("")
lines.append('echo ""')
lines.append('echo "=== Transfer summary ==="')
lines.append('echo "  Succeeded: $OK"')
lines.append('echo "  Failed:    $FAIL"')
lines.append(f'echo "  Total:     {total_ops}"')

rsync_script.write_text("\n".join(lines) + "\n")
rsync_script.chmod(0o755)

print(f"Wrote rsync script: {rsync_script}")
print(f"  MR rsync lines: {len(mr_ops)}")
print(f"  ECC rsync lines: {len(ecc_ops)}")
print(f"\nTo run:  bash {rsync_script}")
print(f"Dry run: see next cell")

Wrote rsync script: /mnt/yeluru/vascular-superenhancement-4d-flow/working_dir/rsync_transfer.sh
  MR rsync lines: 585
  ECC rsync lines: 413

To run:  bash /mnt/yeluru/vascular-superenhancement-4d-flow/working_dir/rsync_transfer.sh
Dry run: see next cell


In [13]:
# === Also write a dry-run version ===
dryrun_script = SCRIPT_DIR / "rsync_transfer_dryrun.sh"

dryrun_lines = [
    "#!/bin/bash",
    "",
    "# Dry-run version — shows what would be transferred without copying anything",
    "",
    "FAIL=0",
    "OK=0",
    "",
    "# === MR exam folder transfers ===",
    "",
]

for dst, srcs in dest_to_sources.items():
    dryrun_lines.append(f"mkdir -p '{dst}'")
    for src in srcs:
        dryrun_lines.append(f"if rsync -avn '{src}/' '{dst}/'; then")
        dryrun_lines.append(f"  ((OK++))")
        dryrun_lines.append(f"else")
        dryrun_lines.append(f"  echo 'FAILED: {src} -> {dst}' >&2")
        dryrun_lines.append(f"  ((FAIL++))")
        dryrun_lines.append(f"fi")

dryrun_lines.append("")
dryrun_lines.append("# === ECC .npy file transfers ===")
dryrun_lines.append("")

for src, dst in ecc_ops:
    dryrun_lines.append(f"if rsync -avn '{src}' '{dst}'; then")
    dryrun_lines.append(f"  ((OK++))")
    dryrun_lines.append(f"else")
    dryrun_lines.append(f"  echo 'FAILED: {src} -> {dst}' >&2")
    dryrun_lines.append(f"  ((FAIL++))")
    dryrun_lines.append(f"fi")

dryrun_lines.append("")
dryrun_lines.append('echo ""')
dryrun_lines.append('echo "=== Dry run summary ==="')
dryrun_lines.append('echo "  Succeeded: $OK"')
dryrun_lines.append('echo "  Failed:    $FAIL"')
dryrun_lines.append(f'echo "  Total:     {len(mr_ops) + len(ecc_ops)}"')

dryrun_script.write_text("\n".join(dryrun_lines) + "\n")
dryrun_script.chmod(0o755)

print(f"Wrote dry-run script: {dryrun_script}")
print(f"\nTo preview: bash {dryrun_script}")

Wrote dry-run script: /mnt/yeluru/vascular-superenhancement-4d-flow/working_dir/rsync_transfer_dryrun.sh

To preview: bash /mnt/yeluru/vascular-superenhancement-4d-flow/working_dir/rsync_transfer_dryrun.sh


In [ ]:
# === Run this AFTER the rsync completes to verify ===
print("=== Post-transfer verification ===")
mr_count = sum(1 for d in MR_DEST.iterdir() if d.is_dir())
ecc_count = sum(1 for f in ECC_DEST.iterdir() if f.is_file())
print(f"MR exam folders on fourier: {mr_count}")
print(f"ECC .npy files on fourier: {ecc_count}")

transferred_mr = set(d.name for d in MR_DEST.iterdir() if d.is_dir())
transferred_ecc = set(f.stem for f in ECC_DEST.iterdir() if f.is_file())

expected_mr = patients.loc[patients["flow4d_src_path"].notna(), "flow4d_pid"]
expected_ecc = patients.loc[patients["ecc_src_path"].notna(), "flow4d_pid"]

missing_mr = expected_mr[~expected_mr.isin(transferred_mr)].tolist()
missing_ecc = expected_ecc[~expected_ecc.isin(transferred_ecc)].tolist()

print(f"\nStill missing MR folders after transfer: {len(missing_mr)}")
if missing_mr:
    print(f"  {missing_mr[:20]}")
print(f"Still missing ECC files after transfer: {len(missing_ecc)}")
if missing_ecc:
    print(f"  {missing_ecc[:20]}")